In [4]:
import numpy as np
import pandas as pd

In [5]:
temp_df = pd.read_csv('/Users/harshraj/Desktop/NLP/data/imdb/IMDB Dataset.csv')

In [6]:
df = temp_df.iloc[:10000]

In [7]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [8]:
df.drop_duplicates(inplace=True)

In [9]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [10]:
df['review'] = df['review'].apply(remove_tags)

In [11]:
df['review'] = df['review'].apply(lambda x:x.lower())

In [12]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

In [13]:
df['review']

0       one reviewers mentioned watching 1 oz episode ...
1       wonderful little production. filming technique...
2       thought wonderful way spend time hot summer we...
3       basically there's family little boy (jake) thi...
4       petter mattei's "love time money" visually stu...
                              ...                        
9995    fun, entertaining movie wwii german spy (julie...
9996    give break. anyone say "good hockey movie"? kn...
9997    movie bad movie. watching endless series bad h...
9998    movie probably made entertain middle school, e...
9999    smashing film film-making. shows intense stran...
Name: review, Length: 9983, dtype: str

In [14]:
import gensim

In [15]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [16]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))
    

In [17]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [18]:
model.build_vocab(story)

In [19]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


(5850136, 6186875)

In [20]:
len(model.wv.index_to_key)

31845

In [21]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [22]:
document_vector(df['review'].values[0])

array([-4.26680315e-03,  3.06783170e-01,  1.19223222e-01, -1.01880386e-01,
       -5.65122478e-02, -4.99627978e-01,  1.38925716e-01,  7.82573938e-01,
       -1.61330551e-01, -1.21514037e-01,  6.26821369e-02, -4.58185017e-01,
        6.23340532e-03,  2.60626405e-01,  5.26561439e-02, -1.59021556e-01,
        1.26937374e-01, -5.27382374e-01,  7.39158913e-02, -7.16862381e-01,
        1.35716334e-01,  6.26511350e-02,  2.26005927e-01, -1.38217822e-01,
       -2.09818766e-01, -1.23739772e-01, -1.71320274e-01, -3.32741946e-01,
       -2.70915061e-01,  1.55793047e-02,  4.96792912e-01,  8.78880247e-02,
        3.84206213e-02, -2.77596444e-01, -3.14719915e-01,  2.57600695e-01,
        1.23647384e-01, -4.60053682e-01, -2.04454631e-01, -8.06903780e-01,
        1.22704782e-01, -3.28936487e-01, -3.32150400e-01, -2.60269605e-02,
        4.66961652e-01, -2.18298510e-01, -4.53926682e-01, -7.61340857e-02,
        1.66754588e-01,  2.68873185e-01,  1.29191428e-01, -2.47488290e-01,
       -2.47476012e-01, -

In [23]:
from tqdm import tqdm

In [24]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

100%|██████████| 9983/9983 [03:05<00:00, 53.96it/s]


In [25]:
X = np.array(X)

In [26]:
X[0]

array([-4.26680315e-03,  3.06783170e-01,  1.19223222e-01, -1.01880386e-01,
       -5.65122478e-02, -4.99627978e-01,  1.38925716e-01,  7.82573938e-01,
       -1.61330551e-01, -1.21514037e-01,  6.26821369e-02, -4.58185017e-01,
        6.23340532e-03,  2.60626405e-01,  5.26561439e-02, -1.59021556e-01,
        1.26937374e-01, -5.27382374e-01,  7.39158913e-02, -7.16862381e-01,
        1.35716334e-01,  6.26511350e-02,  2.26005927e-01, -1.38217822e-01,
       -2.09818766e-01, -1.23739772e-01, -1.71320274e-01, -3.32741946e-01,
       -2.70915061e-01,  1.55793047e-02,  4.96792912e-01,  8.78880247e-02,
        3.84206213e-02, -2.77596444e-01, -3.14719915e-01,  2.57600695e-01,
        1.23647384e-01, -4.60053682e-01, -2.04454631e-01, -8.06903780e-01,
        1.22704782e-01, -3.28936487e-01, -3.32150400e-01, -2.60269605e-02,
        4.66961652e-01, -2.18298510e-01, -4.53926682e-01, -7.61340857e-02,
        1.66754588e-01,  2.68873185e-01,  1.29191428e-01, -2.47488290e-01,
       -2.47476012e-01, -

In [27]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [28]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [29]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [31]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7746619929894842